In [9]:
from pathlib import Path
import json
from pmtiles.writer import write
from pmtiles.tile import zxy_to_tileid, TileType, Compression



def xyz_to_pmtiles(tile_dir, output_path, name):

    tile_dir = Path(tile_dir)
    output_path = Path(output_path)

    # Збираємо координати та шляхи
    tiles = []

    for z_dir in tile_dir.iterdir():

        if not z_dir.is_dir():
            continue

        z = int(z_dir.name)

        for x_dir in z_dir.iterdir():

            if not x_dir.is_dir():
                continue

            x = int(x_dir.name)

            for tile_path in x_dir.glob("*.png"):

                y = int(tile_path.stem)

                tile_id = zxy_to_tileid(z, x, y)

                tiles.append((tile_id, tile_path))

    tiles.sort(key=lambda item: item[0])

    if not tiles:
        print(f"No tiles found: {tile_dir}")
        return

    min_zoom = min(
        int(p.parent.parent.name)
        for _, p in tiles
    )

    max_zoom = max(
        int(p.parent.parent.name)
        for _, p in tiles
    )

    metadata = {
        "name": name,
        "format": "png",
        "type": "overlay",
        "version": "1.0",
        "minzoom": min_zoom,
        "maxzoom": max_zoom
    }

    # Створюємо PMTiles напряму
    with write(str(output_path)) as writer:

        for i, (tile_id, tile_path) in enumerate(tiles, 1):

            with open(tile_path, "rb") as f:
                tile_data = f.read()

            writer.write_tile(tile_id, tile_data)

            if i % 5000 == 0:
                print(f"{name}: {i}/{len(tiles)} tiles")

        writer.finalize(
            {
                "tile_type": TileType.PNG,
                "tile_compression": Compression.NONE,
                "min_zoom": min_zoom,
                "max_zoom": max_zoom,
            },
            metadata
        )

    print(f"Created: {output_path}")

In [2]:
from pathlib import Path
import sqlite3
import subprocess

BASE = Path(r"C:\Users\Sofi\opencv\Scripts\work\germany\Resedium\data")

YEARS = range(2020, 2026)

OUTPUT = BASE / "pmtiles"
OUTPUT.mkdir(exist_ok=True)

TILES = {
    "lclu": lambda year: BASE / "tiles" / f"LULC_class_SachsenAnhalt_{year}",
    "forest": lambda year: BASE / "tiles" / f"Forest_State_{year}",
    "grass": lambda year: BASE / "tiles" / "Permanent_Grassland",
    "grass2": lambda year: BASE / "tiles" / f"Grassland_Conversion",
}

In [10]:
for year in YEARS:

    print(f"\n========== YEAR {year} ==========")

    for layer in ["lclu", "forest"]:

        tile_dir = TILES[layer](year)

        output_path = OUTPUT / f"{tile_dir.name}.pmtiles"

        xyz_to_pmtiles(
            tile_dir,
            output_path,
            tile_dir.name
        )

    if year == 2020:

        for layer in ["grass", "grass2"]:

            tile_dir = TILES[layer](year)

            output_path = OUTPUT / f"{tile_dir.name}.pmtiles"

            xyz_to_pmtiles(
                tile_dir,
                output_path,
                tile_dir.name
            )


========== YEAR 2020 ==========
LULC_class_SachsenAnhalt_2020: 5000/25521 tiles
LULC_class_SachsenAnhalt_2020: 10000/25521 tiles
LULC_class_SachsenAnhalt_2020: 15000/25521 tiles
LULC_class_SachsenAnhalt_2020: 20000/25521 tiles
LULC_class_SachsenAnhalt_2020: 25000/25521 tiles
Created: C:\Users\Sofi\opencv\Scripts\work\germany\Resedium\data\pmtiles\LULC_class_SachsenAnhalt_2020.pmtiles
Forest_State_2020: 5000/27263 tiles
Forest_State_2020: 10000/27263 tiles
Forest_State_2020: 15000/27263 tiles
Forest_State_2020: 20000/27263 tiles
Forest_State_2020: 25000/27263 tiles
Created: C:\Users\Sofi\opencv\Scripts\work\germany\Resedium\data\pmtiles\Forest_State_2020.pmtiles
Permanent_Grassland: 5000/25521 tiles
Permanent_Grassland: 10000/25521 tiles
Permanent_Grassland: 15000/25521 tiles
Permanent_Grassland: 20000/25521 tiles
Permanent_Grassland: 25000/25521 tiles
Created: C:\Users\Sofi\opencv\Scripts\work\germany\Resedium\data\pmtiles\Permanent_Grassland.pmtiles
Grassland_Conversion: 5000/25521 t

In [10]:
print(mbtiles_path, pmtiles_path)

C:\Users\Sofi\opencv\Scripts\work\germany\Resedium\data\pmtiles\LULC_class_SachsenAnhalt_2020.mbtiles C:\Users\Sofi\opencv\Scripts\work\germany\Resedium\data\pmtiles\LULC_class_SachsenAnhalt_2020.pmtiles


In [11]:
!pip install pmtiles


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pmtiles
import inspect
from pmtiles.writer import Writer

print("Version:", getattr(pmtiles, "__version__", "unknown"))
print(inspect.signature(Writer.finalize))

Version: unknown
(self, header, metadata)
